# Evaluate a Trained M-CBM

In [ ]:
from pathlib import Path
import json
import random
import re
import os
import sys
from io import BytesIO

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image as IPythonImage
from PIL import Image

from mcbm.data.utils import get_classes, get_image_path
from mcbm.scripts.name_concepts import load_activation_rows
from mcbm.scripts.ncc_evaluation import build_concept_logits, load_config_for_cbm
from mcbm.scripts.train_cbm import normalize_concept_features


## Select Run

In [ ]:
cbm_dir = PROJECT_DIR / "outputs/cbm/imagenet/cbm_imagenet_20260713_123602"
ncc_target = 5
use_ncc = True
device = "cuda" if torch.cuda.is_available() else "cpu"


## Load Saved Files

In [ ]:
def display_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    display(IPythonImage(data=buffer.getvalue()))

def project_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_DIR / path

config, _, run_info = load_config_for_cbm(cbm_dir)
dataset = config["dataset"]
activation_dir = project_path(run_info["activation_dir"])

concepts = [line.strip() for line in (cbm_dir / "concepts.txt").read_text().splitlines() if line.strip()]
classes = [name for name in get_classes(dataset) if name]

train_logits, _, val_logits, _, test_logits, test_labels = build_concept_logits(
    cbm_dir, config, run_info, torch.device(device)
)
_, _, normalized_features = normalize_concept_features(train_logits, val_logits, test_logits)
test_features = normalized_features[2]
test_confidences = torch.sigmoid(test_logits)
_, test_image_names, _ = load_activation_rows(activation_dir / "activations_test.pt")

if use_ncc:
    weight_path = cbm_dir / "ncc" / f"W_g@NCC_target={ncc_target}.pt"
    bias_path = cbm_dir / "ncc" / f"b_g@NCC_target={ncc_target}.pt"
    if not weight_path.exists() or not bias_path.exists():
        raise FileNotFoundError(f"NCC@{ncc_target} weights not found. Run ncc_evaluation.py first, or set use_ncc = False.")
    final_weight = torch.load(weight_path, map_location="cpu").float()
    final_bias = torch.load(bias_path, map_location="cpu").float()
    model_name = f"NCC@{ncc_target}"
else:
    state = torch.load(cbm_dir / "final.pt", map_location="cpu")
    final_weight = state["weight"].float()
    final_bias = state["bias"].float()
    model_name = "SAGA final layer"

print(f"Dataset: {dataset}")
print(f"Model: {model_name}")
print(f"Concepts: {len(concepts)}")
print(f"Classes: {len(classes)}")
print(f"Test examples: {len(test_labels)}")


## Run Summary

In [ ]:
def flatten_dict(values, prefix=""):
    rows = []
    for key, value in values.items():
        name = f"{prefix}.{key}" if prefix else key
        if name.startswith("per_class_accuracies"):
            continue
        if isinstance(value, dict):
            rows.extend(flatten_dict(value, name))
        else:
            rows.append({"metric": name, "value": value})
    return rows

metrics_path = cbm_dir / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics_df = pd.DataFrame(flatten_dict(metrics))
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(metrics_df)

ncc_table_path = cbm_dir / "ncc" / "ncc_selected_models.csv"
if ncc_table_path.exists():
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
        display(pd.read_csv(ncc_table_path))


## Test Accuracy

In [ ]:
def balanced_accuracy(preds, labels, num_classes):
    rows = labels.cpu() * num_classes + preds.cpu()
    confusion = torch.bincount(rows, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    support = confusion.sum(dim=1)
    recall = confusion.diag().float() / support.clamp_min(1).float()
    return recall[support > 0].mean().item()

with torch.no_grad():
    logits = test_features @ final_weight.T + final_bias
    preds = logits.argmax(dim=1)

accuracy = (preds == test_labels).float().mean().item()
bal_acc = balanced_accuracy(preds, test_labels, len(classes))

print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {bal_acc:.4f}")

## Class Concept Weights

In [ ]:
def find_class_index(query):
    if isinstance(query, int):
        return query

    query = query.lower()
    matches = [i for i, name in enumerate(classes) if query in name.lower()]
    if not matches:
        raise ValueError(f"No class matched: {query}")
    if len(matches) > 1:
        print("Multiple matches:")
        for i in matches[:20]:
            print(i, classes[i])
        raise ValueError("Use a more specific query.")
    return matches[0]


def show_class_weights(class_query, k=10, min_abs_weight=1e-8):
    class_id = find_class_index(class_query)
    weights = final_weight[class_id].detach().cpu()
    nonzero = torch.nonzero(weights.abs() > min_abs_weight).flatten()

    print(f"Class {class_id}: {classes[class_id]}")
    if len(nonzero) == 0:
        return pd.DataFrame(columns=["rank", "concept_id", "concept", "weight"])

    order = torch.argsort(weights[nonzero].abs(), descending=True)
    selected = nonzero[order[:min(k, len(order))]]

    rows = []
    for rank, concept_id in enumerate(selected.tolist(), start=1):
        rows.append({
            "rank": rank,
            "concept_id": concept_id,
            "concept": concepts[concept_id],
            "weight": float(weights[concept_id]),
        })
    return pd.DataFrame(rows)

used = (final_weight.abs().sum(dim=0) > 1e-8).sum().item()
print(f"Used concepts: {used}/{len(concepts)}")

show_class_weights(0, k=5)


## Top Activating Images


In [ ]:
def find_concept_index(query):
    if isinstance(query, int):
        return query

    query = query.lower()
    exact = [i for i, name in enumerate(concepts) if name.lower() == query]
    if len(exact) == 1:
        return exact[0]

    matches = [i for i, name in enumerate(concepts) if query in name.lower()]
    if not matches:
        raise ValueError(f"No concept matched: {query}")
    if len(matches) > 1:
        print("Multiple matches:")
        for i in matches[:30]:
            print(i, concepts[i])
        raise ValueError("Use a more specific query.")
    return matches[0]


def top_images_for_concept(concept_query, k=5):
    concept_id = find_concept_index(concept_query)
    values, indices = torch.topk(test_confidences[:, concept_id], k=min(k, test_features.shape[0]))
    rows = []
    for rank, (value, index) in enumerate(zip(values, indices), start=1):
        image_name = test_image_names[int(index)]
        rows.append({
            "rank": rank,
            "test_index": int(index),
            "image_name": image_name,
            "class": classes[int(test_labels[int(index)])],
            "confidence": float(value),
        })
    print(f"Concept {concept_id}: {concepts[concept_id]}")
    return pd.DataFrame(rows)


def show_top_images_for_concept(concept_query, k=5, columns=5):
    table = top_images_for_concept(concept_query, k=k)
    display(table)

    images = []
    for _, row in table.iterrows():
        image_path = get_image_path(dataset, "test", row["image_name"])
        if image_path.exists():
            images.append(Image.open(image_path).convert("RGB"))
        else:
            print(f"Missing image: {image_path}")

    if not images:
        return

    columns = min(columns, len(images))
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 3 * rows))
    axes = np.atleast_1d(axes).reshape(rows, columns)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, image in zip(axes.ravel(), images):
        ax.imshow(image)

    plt.tight_layout()
    display_figure(fig)

concept_query = 0  # can also be an integer concept id, e.g. 0
show_top_images_for_concept(concept_query, k=5)


## Random Test Explanations


In [ ]:
red_rgb = np.array([1.0, 0.0, 0.31796406298163893])
blue_rgb = np.array([0.0, 0.5433775692459107, 0.9833790623014013])


def format_value(s, format_str):
    if not issubclass(type(s), str):
        s = format_str % s
    s = re.sub(r"\.?0+$", "", s)
    if s[0] == "-":
        s = "\u2212" + s[1:]
    return s


def bar(contributions, feature_names, max_display=10, show=True, title=None, fontsize=13):
    values = contributions.copy()
    xlabel = "Concept contributions"

    if max_display is None:
        max_display = len(feature_names)
    num_features = min(max_display, len(values))
    max_display = min(max_display, num_features)

    orig_inds = [i for i in range(len(values))]
    feature_order = np.argsort(np.abs(values), 0)[::-1]
    feature_inds = feature_order[:max_display]
    y_pos = np.arange(len(feature_inds), 0, -1)

    feature_names_new = []
    for inds in orig_inds:
        feature_names_new.append(feature_names[inds])
    feature_names = feature_names_new

    yticklabels = []
    for i in feature_inds:
        yticklabels.append(feature_names[i])

    if num_features < len(values):
        num_cut = np.sum([1 for i in range(num_features - 1, len(values))])
        values[feature_order[num_features - 1]] = np.sum(
            [values[feature_order[i]] for i in range(num_features - 1, len(values))], 0
        )
        yticklabels[-1] = "Sum of %d other features" % num_cut

    row_height = 0.55
    plt.gcf().set_size_inches(8, num_features * row_height + 1.5)

    negative_values_present = np.sum(values[feature_order[:num_features]] < 0) > 0
    if negative_values_present:
        plt.axvline(0, 0, 1, color="#000000", linestyle="-", linewidth=1, zorder=1)

    bar_width = 0.7

    plt.barh(
        y_pos,
        values[feature_inds],
        bar_width,
        align="center",
        color=[blue_rgb if values[feature_inds[j]] <= 0 else red_rgb for j in range(len(y_pos))],
        hatch=None,
        edgecolor=(1, 1, 1, 0.8),
        label=None,
    )

    plt.yticks(
        list(y_pos) + list(y_pos + 1e-8),
        yticklabels + [label.split("=")[-1] for label in yticklabels],
        fontsize=fontsize,
    )

    xlen = plt.xlim()[1] - plt.xlim()[0]
    fig = plt.gcf()
    ax = plt.gca()
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    width = bbox.width
    bbox_to_xscale = xlen / width

    for j in range(len(y_pos)):
        ind = feature_order[j]
        if values[ind] < 0:
            plt.text(
                values[ind] - (5 / 72) * bbox_to_xscale,
                y_pos[j],
                format_value(values[ind], "%+0.02f"),
                horizontalalignment="right",
                verticalalignment="center",
                color=blue_rgb,
                fontsize=fontsize,
            )
        else:
            plt.text(
                values[ind] + (5 / 72) * bbox_to_xscale,
                y_pos[j],
                format_value(values[ind], "%+0.02f"),
                horizontalalignment="left",
                verticalalignment="center",
                color=red_rgb,
                fontsize=fontsize,
            )

    for i in range(num_features):
        plt.axhline(i + 1, color="#888888", lw=0.5, dashes=(1, 5), zorder=-1)

    plt.gca().xaxis.set_ticks_position("bottom")
    plt.gca().yaxis.set_ticks_position("none")
    plt.gca().spines["right"].set_visible(False)
    plt.gca().spines["top"].set_visible(False)
    if negative_values_present:
        plt.gca().spines["left"].set_visible(False)
    plt.gca().tick_params("x", labelsize=fontsize)

    xmin, xmax = plt.gca().get_xlim()
    if negative_values_present:
        plt.gca().set_xlim(xmin - (xmax - xmin) * 0.1, xmax + (xmax - xmin) * 0.1)
    else:
        plt.gca().set_xlim(xmin, xmax + (xmax - xmin) * 0.1)

    plt.xlabel(xlabel, fontsize=fontsize)
    if title:
        plt.title(title, fontsize=fontsize)

    if show:
        display_figure(plt.gcf())


def explain_test_example(test_index):
    image_name = test_image_names[test_index]
    image_path = get_image_path(dataset, "test", image_name)
    if image_path.exists():
        display(Image.open(image_path).convert("RGB").resize([320, 320]))
    else:
        print(f"Missing image: {image_path}")

    concept_act = test_features[test_index].detach().cpu()
    concept_confidence = test_confidences[test_index].detach().cpu()
    outputs = concept_act @ final_weight.detach().cpu().T + final_bias.detach().cpu()

    top_logit_vals, top_classes = torch.topk(outputs, dim=0, k=2)
    conf = torch.nn.functional.softmax(outputs, dim=0)
    label = int(test_labels[test_index])
    print(
        "Image:{} Gt:{}, 1st Pred:{}, {:.3f}, 2nd Pred:{}, {:.3f}".format(
            test_index,
            classes[label],
            classes[int(top_classes[0])],
            top_logit_vals[0],
            classes[int(top_classes[1])],
            top_logit_vals[1],
        )
    )

    for k in range(1):
        contributions = concept_act * final_weight[int(top_classes[k]), :].detach().cpu()
        weight_vector = final_weight[int(top_classes[k]), :].detach().cpu()
        abs_contributions = torch.abs(contributions.cpu())

        mask = (abs_contributions < 0.01) & (weight_vector > 0)
        small_pos_weight_contributions = mask.sum().item()
        print(f"Concepts with weight > 0 and contribution < 1e-2: {small_pos_weight_contributions}")

        feature_names = [("NOT " if concept_confidence[i] < 0.5 else "") + concepts[i] for i in range(len(concepts))]
        values = contributions.cpu().numpy()
        max_display = min(int(sum(abs(values) > 0.005)) + 1, 8)

        print("Concept confidence for shown concepts:")
        shown_indices = torch.argsort(abs_contributions, descending=True)[:max_display - 1].tolist()
        for idx in shown_indices:
            confidence = concept_confidence[idx] if concept_confidence[idx] >= 0.5 else 1 - concept_confidence[idx]
            print(f"{feature_names[idx]}: confidence={confidence:.3f}")

        title = "Pred:{} - Conf: {:.3f} - Logit:{:.2f} - Bias:{:.2f}".format(
            classes[int(top_classes[k])],
            conf[int(top_classes[k])],
            top_logit_vals[k],
            final_bias[int(top_classes[k])],
        )
        bar(values, feature_names, max_display=max_display, title=title, fontsize=16)


def explain_random_test_examples(n=5, seed=0):
    to_display = random.Random(seed).sample([i for i in range(len(test_labels))], k=min(n, len(test_labels)))
    for test_index in to_display:
        explain_test_example(test_index)

explain_random_test_examples(n=5, seed=0)
